In [1]:
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

camera_is_ready = False

if "camera" in globals():
    try:
        camera_is_ready = (
            hasattr(camera, "cap")
            and camera.cap.isOpened()
        )
    except Exception:
        camera_is_ready = False

if camera_is_ready:
    camera.running = True
    print("✅ Camera đã tồn tại — tiếp tục sử dụng camera cũ")
else:
    camera = CSICamera(
        width=224,
        height=224,
        capture_width=1280,
        capture_height=720,
        capture_fps=30
    )

    camera.running = True
    print("✅ Đã khởi tạo camera mới")

✅ Đã khởi tạo camera mới


In [2]:
# ========================================================
# CAMERA LIVE + RECORD DATASET
# Chạy sau cell khởi tạo car và controller
# ========================================================

import os

# Chỉ áp dụng tạm thời trong kernel hiện tại
os.environ.setdefault("OPENBLAS_CORETYPE", "ARMV8")

import csv
import cv2
import time
import threading

from pathlib import Path
from datetime import datetime
import ipywidgets.widgets as widgets
from jetcam.csi_camera import CSICamera


# ========================================================
# 1. CẤU HÌNH
# ========================================================
DATASET_ROOT = Path("/home/jetson/dataset_steering")

# Một mẫu mỗi giây
RECORD_INTERVAL_SECONDS = 1.0

# Chất lượng ảnh dataset
DATASET_JPEG_QUALITY = 85

# Hiển thị camera khoảng 12 FPS
PREVIEW_FPS = 12
PREVIEW_JPEG_QUALITY = 55

# Góc lái ước tính tại car.steering = ±1
STEERING_MAX_DEG = 30.0


# ========================================================
# 2. DỌN THREAD VÀ CALLBACK CŨ KHI CHẠY LẠI CELL
# ========================================================
try:
    _dataset_stop_event.set()
except Exception:
    pass

try:
    if (
        _dataset_thread is not None
        and _dataset_thread.is_alive()
    ):
        _dataset_thread.join(timeout=2.0)
except Exception:
    pass

try:
    camera.unobserve(
        _update_camera_preview,
        names="value"
    )
except Exception:
    pass

# Vô hiệu hóa nút của output cũ
try:
    btn_record_start.disabled = True
    btn_record_stop.disabled = True
except Exception:
    pass


# ========================================================
# 3. KHỞI TẠO HOẶC TÁI SỬ DỤNG CAMERA
# ========================================================
camera_ready = False

if "camera" in globals():
    try:
        camera_ready = (
            hasattr(camera, "cap")
            and camera.cap.isOpened()
        )
    except Exception:
        camera_ready = False

if camera_ready:
    print("✅ Tái sử dụng camera đang có")
else:
    camera = CSICamera(
        width=224,
        height=224,
        capture_width=1280,
        capture_height=720,
        capture_fps=30
    )

    print("✅ Đã khởi tạo camera mới")


# ========================================================
# 4. HIỂN THỊ CAMERA LIVE
# ========================================================
camera_view = widgets.Image(
    format="jpeg",
    width=320,
    height=320
)

preview_status = widgets.HTML(
    value="<b>📷 Camera:</b> Đang khởi động..."
)

_preview_state = {
    "last_update": 0.0
}

_preview_interval = 1.0 / PREVIEW_FPS


def _update_camera_preview(change):
    now = time.monotonic()

    if now - _preview_state["last_update"] < _preview_interval:
        return

    frame = change["new"]

    if frame is None:
        return

    success, jpeg = cv2.imencode(
        ".jpg",
        frame,
        [
            int(cv2.IMWRITE_JPEG_QUALITY),
            PREVIEW_JPEG_QUALITY
        ]
    )

    if success:
        camera_view.value = jpeg.tobytes()
        _preview_state["last_update"] = now


camera.observe(
    _update_camera_preview,
    names="value"
)

camera.running = True

preview_status.value = (
    "<b style='color:green'>📷 Camera đang hoạt động</b>"
    " — Preview khoảng {} FPS".format(PREVIEW_FPS)
)


# ========================================================
# 5. TRẠNG THÁI RECORD
# ========================================================
_dataset_stop_event = threading.Event()
_dataset_thread = None

_dataset_state = {
    "count": 0,
    "session_dir": None
}


# ========================================================
# 6. GIAO DIỆN RECORD
# ========================================================
btn_record_start = widgets.Button(
    description="🔴 BẬT RECORD",
    button_style="success",
    layout=widgets.Layout(
        width="210px",
        height="42px"
    )
)

btn_record_stop = widgets.Button(
    description="⏹ TẮT RECORD",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(
        width="210px",
        height="42px"
    )
)

record_status = widgets.HTML(
    value="<b>⚪ Record:</b> Chưa ghi dataset"
)

record_latest = widgets.HTML(
    value="<b>Mẫu gần nhất:</b> Chưa có"
)

record_output = widgets.Output()


# ========================================================
# 7. THREAD GHI ẢNH VÀ LOG CSV
# ========================================================
def _dataset_record_worker(
    session_dir,
    stop_event
):
    images_dir = session_dir / "images"
    csv_path = session_dir / "labels.csv"

    start_time = time.monotonic()
    next_sample_time = start_time
    sample_index = 0

    try:
        with csv_path.open("w", newline="") as csv_file:
            writer = csv.writer(csv_file)

            writer.writerow([
                "sample_index",
                "timestamp",
                "elapsed_seconds",
                "image_file",
                "steering_normalized",
                "steering_angle_est_deg",
                "steering_gain",
                "throttle_normalized",
                "throttle_gain"
            ])

            csv_file.flush()

            while not stop_event.is_set():
                wait_seconds = (
                    next_sample_time
                    - time.monotonic()
                )

                if wait_seconds > 0:
                    if stop_event.wait(wait_seconds):
                        break

                try:
                    frame = getattr(camera, "value", None)

                    if frame is None:
                        raise RuntimeError(
                            "Camera chưa có frame."
                        )

                    # Sao chép frame mới nhất
                    frame_copy = frame.copy()

                    steering_value = float(car.steering)
                    throttle_value = float(car.throttle)

                    steering_value = max(
                        -1.0,
                        min(1.0, steering_value)
                    )

                    steering_angle_deg = (
                        steering_value
                        * STEERING_MAX_DEG
                    )

                    timestamp = datetime.now().isoformat(
                        timespec="milliseconds"
                    )

                    elapsed_seconds = (
                        time.monotonic()
                        - start_time
                    )

                    image_name = (
                        "frame_{:06d}.jpg"
                        .format(sample_index)
                    )

                    relative_image_path = os.path.join(
                        "images",
                        image_name
                    )

                    absolute_image_path = (
                        images_dir / image_name
                    )

                    saved = cv2.imwrite(
                        str(absolute_image_path),
                        frame_copy,
                        [
                            int(cv2.IMWRITE_JPEG_QUALITY),
                            DATASET_JPEG_QUALITY
                        ]
                    )

                    if not saved:
                        raise RuntimeError(
                            "Không thể lưu {}".format(
                                image_name
                            )
                        )

                    writer.writerow([
                        sample_index,
                        timestamp,
                        "{:.3f}".format(elapsed_seconds),
                        relative_image_path,
                        "{:.6f}".format(steering_value),
                        "{:.3f}".format(
                            steering_angle_deg
                        ),
                        "{:.3f}".format(
                            float(
                                getattr(
                                    car,
                                    "steering_gain",
                                    1.0
                                )
                            )
                        ),
                        "{:.6f}".format(throttle_value),
                        "{:.3f}".format(
                            float(
                                getattr(
                                    car,
                                    "throttle_gain",
                                    1.0
                                )
                            )
                        )
                    ])

                    csv_file.flush()

                    sample_index += 1
                    _dataset_state["count"] = sample_index

                    record_status.value = (
                        "<b style='color:red'>"
                        "🔴 ĐANG RECORD"
                        "</b> — {} mẫu".format(
                            sample_index
                        )
                    )

                    record_latest.value = (
                        "<b>Mẫu gần nhất:</b> {}"
                        " | Lái: {:.3f}"
                        " | Góc: {:.1f}°"
                        " | Ga: {:.3f}"
                        .format(
                            image_name,
                            steering_value,
                            steering_angle_deg,
                            throttle_value
                        )
                    )

                except Exception as error:
                    record_latest.value = (
                        "<b style='color:red'>"
                        "⚠️ Lỗi ghi mẫu:"
                        "</b> {}".format(error)
                    )

                next_sample_time += (
                    RECORD_INTERVAL_SECONDS
                )

                # Không ghi dồn nhiều frame nếu Jetson bị chậm
                if next_sample_time < time.monotonic():
                    next_sample_time = (
                        time.monotonic()
                        + RECORD_INTERVAL_SECONDS
                    )

    except Exception as error:
        record_status.value = (
            "<b style='color:red'>"
            "❌ RECORD BỊ DỪNG:"
            "</b> {}".format(error)
        )

        btn_record_start.disabled = False
        btn_record_stop.disabled = True


# ========================================================
# 8. BẬT RECORD
# ========================================================
def bat_record(b):
    global _dataset_thread
    global _dataset_stop_event

    with record_output:
        record_output.clear_output()

        try:
            if "car" not in globals():
                raise RuntimeError(
                    "Chưa có biến car. "
                    "Hãy chạy cell điều khiển xe trước."
                )

            if (
                _dataset_thread is not None
                and _dataset_thread.is_alive()
            ):
                print("⚠️ Dataset đang được ghi.")
                return

            if getattr(camera, "value", None) is None:
                raise RuntimeError(
                    "Camera chưa cung cấp hình ảnh."
                )

            session_name = datetime.now().strftime(
                "session_%Y%m%d_%H%M%S_%f"
            )

            session_dir = (
                DATASET_ROOT / session_name
            )

            images_dir = (
                session_dir / "images"
            )

            images_dir.mkdir(
                parents=True,
                exist_ok=False
            )

            _dataset_state["count"] = 0
            _dataset_state["session_dir"] = str(
                session_dir
            )

            _dataset_stop_event = threading.Event()

            _dataset_thread = threading.Thread(
                target=_dataset_record_worker,
                args=(
                    session_dir,
                    _dataset_stop_event
                ),
                daemon=True
            )

            _dataset_thread.start()

            btn_record_start.disabled = True
            btn_record_stop.disabled = False

            record_status.value = (
                "<b style='color:red'>"
                "🔴 ĐANG RECORD"
                "</b>"
            )

            record_latest.value = (
                "<b>Mẫu gần nhất:</b> "
                "Đang chờ mẫu đầu tiên..."
            )

            print("✅ Bắt đầu ghi dataset")
            print("📁", session_dir)
            print("⏱️ Một ảnh mỗi giây")
            print("📄", session_dir / "labels.csv")

        except Exception as error:
            record_status.value = (
                "<b style='color:red'>"
                "❌ Không thể record:"
                "</b> {}".format(error)
            )

            print("❌", error)


# ========================================================
# 9. TẮT RECORD
# ========================================================
def tat_record(b):
    global _dataset_thread

    _dataset_stop_event.set()

    if (
        _dataset_thread is not None
        and _dataset_thread.is_alive()
    ):
        _dataset_thread.join(timeout=2.0)

    btn_record_start.disabled = False
    btn_record_stop.disabled = True

    record_status.value = (
        "<b>⏹ ĐÃ DỪNG RECORD</b>"
        " — Tổng cộng {} mẫu".format(
            _dataset_state["count"]
        )
    )

    with record_output:
        print("⏹ Đã dừng record")
        print(
            "🖼️ Tổng số ảnh:",
            _dataset_state["count"]
        )
        print(
            "📁 Dataset:",
            _dataset_state["session_dir"]
        )


btn_record_start.on_click(bat_record)
btn_record_stop.on_click(tat_record)


# ========================================================
# 10. HIỂN THỊ CAMERA VÀ RECORD
# ========================================================
display(
    widgets.VBox([
        widgets.HTML(
            value="<h3>Camera và Dataset Recorder</h3>"
        ),
        camera_view,
        preview_status,
        widgets.HBox([
            btn_record_start,
            btn_record_stop
        ]),
        record_status,
        record_latest,
        record_output
    ])
)

✅ Tái sử dụng camera đang có


In [3]:
import traitlets
import ipywidgets.widgets as widgets
from jetracer.nvidia_racecar import NvidiaRacecar


# ========================================================
# 1. KHỞI TẠO XE
# ========================================================
car = NvidiaRacecar()

# Đảm bảo xe đứng yên khi khởi tạo
car.throttle = 0
car.steering = 0

try:
    car.steering_motor.set_pulse_width_range(500, 2500)
    print("✅ Đã thiết lập PWM lái: 500–2500")
except Exception as e:
    print("⚠️ Không thể thiết lập PWM lái:", e)

# Độ nhạy tay lái
car.steering_gain = -1.0

# Giới hạn ga ban đầu: 90%
# Có thể điều chỉnh từ 50% đến 100% bằng thanh trượt bên dưới
car.throttle_gain = 0.9

print(f"⚡ Giới hạn ga ban đầu: {car.throttle_gain * 100:.0f}%")


# ========================================================
# 2. KHỞI TẠO TAY CẦM
# ========================================================
controller = widgets.Controller(index=0)

print("👉 Bấm một nút hoặc xoay cả hai cần gạt để kích hoạt tay cầm.")
display(controller)


# ========================================================
# 3. THANH ĐIỀU CHỈNH TỐC ĐỘ
# ========================================================
speed_slider = widgets.FloatSlider(
    value=0.9,
    min=0.5,
    max=1.0,
    step=0.05,
    description="Tốc độ:",
    readout=True,
    readout_format=".0%",
    continuous_update=False,
    layout=widgets.Layout(width="420px")
)


# ========================================================
# 4. NÚT KẾT NỐI VÀ DỪNG KHẨN CẤP
# ========================================================
btn_connect = widgets.Button(
    description="KÍCH HOẠT ĐIỀU KHIỂN XE",
    button_style="success",
    layout=widgets.Layout(width="300px", height="40px")
)

btn_stop = widgets.Button(
    description="DỪNG KHẨN CẤP",
    button_style="danger",
    layout=widgets.Layout(width="220px", height="40px")
)

output = widgets.Output()

# Lưu liên kết để tránh tạo nhiều liên kết khi bấm nút nhiều lần
steering_link = None
throttle_link = None


# ========================================================
# 5. HÀM BIẾN ĐỔI TAY CẦM
# ========================================================
def dieu_khien_lai(x):
    # Dead-zone giúp xe không tự bẻ lái khi cần ở gần tâm
    if abs(x) < 0.05:
        return 0.0

    return max(-1.0, min(1.0, x))


def dieu_khien_ga(x):
    # Đảo chiều trục: đẩy cần lên để chạy tiến
    value = -x

    # Dead-zone giúp xe không tự chạy khi cần ở gần tâm
    if abs(value) < 0.08:
        return 0.0

    return max(-1.0, min(1.0, value))


# ========================================================
# 6. CẬP NHẬT GIỚI HẠN TỐC ĐỘ
# ========================================================
def cap_nhat_toc_do(change):
    # Dừng xe trước khi thay đổi giới hạn ga
    car.throttle = 0
    car.throttle_gain = change["new"]

    with output:
        print(f"⚡ Đã đặt giới hạn ga: {car.throttle_gain * 100:.0f}%")


speed_slider.observe(cap_nhat_toc_do, names="value")


# ========================================================
# 7. KẾT NỐI TAY CẦM VỚI XE
# ========================================================
def thuc_hien_ket_noi(b):
    global steering_link, throttle_link

    with output:
        output.clear_output()

        if len(controller.axes) < 3:
            print("❌ Trình duyệt chưa nhận diện đủ các trục tay cầm.")
            print("👉 Hãy xoay tròn cả hai cần gạt rồi bấm lại.")
            return

        try:
            # Xóa liên kết cũ nếu đã từng kết nối
            if steering_link is not None:
                steering_link.unlink()

            if throttle_link is not None:
                throttle_link.unlink()

            car.throttle = 0
            car.steering = 0

            # Cần phải: điều khiển lái trái/phải
            steering_link = traitlets.dlink(
                (controller.axes[2], "value"),
                (car, "steering"),
                transform=dieu_khien_lai
            )

            # Cần trái: điều khiển tiến/lùi
            throttle_link = traitlets.dlink(
                (controller.axes[1], "value"),
                (car, "throttle"),
                transform=dieu_khien_ga
            )

            print("🚀 Đã kích hoạt điều khiển xe!")
            print("🕹️ Cần TRÁI: Tiến/Lùi")
            print("🕹️ Cần PHẢI: Bẻ lái")
            print(f"⚡ Giới hạn ga: {car.throttle_gain * 100:.0f}%")

        except Exception as error:
            car.throttle = 0
            print(f"❌ Không thể thiết lập điều khiển: {error}")


# ========================================================
# 8. DỪNG KHẨN CẤP
# ========================================================
def dung_khan_cap(b):
    global steering_link, throttle_link

    try:
        if throttle_link is not None:
            throttle_link.unlink()
            throttle_link = None

        if steering_link is not None:
            steering_link.unlink()
            steering_link = None

        car.throttle = 0
        car.steering = 0

        with output:
            output.clear_output()
            print("🛑 ĐÃ DỪNG XE VÀ NGẮT LIÊN KẾT TAY CẦM")

    except Exception as error:
        with output:
            print(f"⚠️ Lỗi khi dừng xe: {error}")


btn_connect.on_click(thuc_hien_ket_noi)
btn_stop.on_click(dung_khan_cap)


# ========================================================
# 9. HIỂN THỊ GIAO DIỆN
# ========================================================
print("-" * 60)

display(
    widgets.VBox([
        speed_slider,
        widgets.HBox([btn_connect, btn_stop]),
        output
    ])
)

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


✅ Đã thiết lập PWM lái: 500–2500
⚡ Giới hạn ga ban đầu: 90%
👉 Bấm một nút hoặc xoay cả hai cần gạt để kích hoạt tay cầm.


Controller()

------------------------------------------------------------
